# Module 08-3: Retry + Circuit Breaker 통합 에이전트

## 학습 목표
- `@retry_on_llm_error` 데코레이터를 LangGraph 노드에 적용할 수 있다
- Circuit Breaker로 외부 API 호출을 보호하는 노드를 구현할 수 있다
- 에러 라우팅 함수로 성공/실패 경로를 분기하는 3노드 에이전트를 구성할 수 있다

**참조 문서**: `docs/part4-production/08-error-handling-resilience.md` 섹션 2.3, 3

## 개념 요약

### 전체 에이전트 흐름

```
[fetch_data] ──> [analyze] ──> [create_ticket]
 (외부 API)       (LLM)         (외부 API)
     │                │              │
     ▼                ▼              ▼
  Circuit           Retry         Circuit
  Breaker          Decorator      Breaker
     │                │              │
     └────────────────┴──────────────┘
                       │
              에러 발생 시 state["error"] 설정
                       │
              [handle_error] 노드로 라우팅
```

### 핵심 패턴

| 패턴 | 적용 대상 | 효과 |
|------|----------|------|
| `@retry_on_llm_error` 데코레이터 | LLM 호출 노드 | Retriable 에러 자동 재시도 |
| `circuit.call()` | 외부 API 호출 노드 | 연속 실패 시 빠른 차단 |
| `route_on_error()` 라우터 | 조건부 엣지 | error 필드로 경로 분기 |

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import time
import random
import threading
import logging
from enum import Enum
from dataclasses import dataclass, field
from functools import wraps
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

logging.basicConfig(level=logging.WARNING)
print("환경 설정 완료")

In [ ]:
# 앞서 구현한 CircuitBreaker 재사용 (Module 08-2에서 가져옴)

class CircuitState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"


class CircuitOpenError(Exception):
    def __init__(self, name: str, remaining_seconds: float):
        self.name = name
        self.remaining_seconds = remaining_seconds
        super().__init__(f"Circuit '{name}'이 OPEN 상태입니다. {remaining_seconds:.1f}초 후 재시도.")


@dataclass
class CircuitBreaker:
    name: str
    failure_threshold: int = 5
    recovery_timeout: float = 60.0
    expected_exceptions: tuple = (Exception,)

    _state: CircuitState = field(default=CircuitState.CLOSED, init=False)
    _failure_count: int = field(default=0, init=False)
    _last_failure_time: float = field(default=0.0, init=False)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    @property
    def state(self):
        with self._lock:
            if self._state == CircuitState.OPEN:
                if time.monotonic() - self._last_failure_time >= self.recovery_timeout:
                    self._state = CircuitState.HALF_OPEN
            return self._state

    def _record_success(self):
        with self._lock:
            self._failure_count = 0
            if self._state == CircuitState.HALF_OPEN:
                self._state = CircuitState.CLOSED

    def _record_failure(self):
        with self._lock:
            self._failure_count += 1
            self._last_failure_time = time.monotonic()
            if self._failure_count >= self.failure_threshold:
                self._state = CircuitState.OPEN

    def call(self, fn, *args, **kwargs):
        if self.state == CircuitState.OPEN:
            remaining = self.recovery_timeout - (time.monotonic() - self._last_failure_time)
            raise CircuitOpenError(self.name, max(0, remaining))
        try:
            result = fn(*args, **kwargs)
            self._record_success()
            return result
        except self.expected_exceptions:
            self._record_failure()
            raise


print("CircuitBreaker 로드 완료")

## 실습 1: @retry_on_llm_error 데코레이터 구현

LangGraph 노드 함수에 적용하는 재시도 데코레이터를 구현합니다.

- **Retriable 에러** (ConnectionError, TimeoutError 등) → backoff 재시도
- **Non-retriable 에러** (ValueError 등) → `state["error"]`에 기록하고 반환
- **모든 재시도 실패** → `state["error"]`에 기록 (DLQ 직행 방지)

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): 2중 데코레이터 구조를 사용합니다.
#               outer function (파라미터 받음) → decorator (함수 받음) → wrapper (실행)
#               RETRIABLE_EXCEPTIONS 튜플에 포함된 예외는 재시도,
#               그 외 예외는 즉시 {"error": ...} dict를 반환하세요.
#
# 힌트 2 (키워드): @wraps(node_fn), RETRIABLE_EXCEPTIONS, functools.wraps,
#               return {"error": f"[{node_fn.__name__}] ...", "current_step": node_fn.__name__}
#
# 힌트 3 (거의 정답):
#   def retry_on_llm_error(max_retries=3, base_delay=1.0, max_delay=30.0, backoff_factor=2.0):
#       def decorator(node_fn):
#           @wraps(node_fn)
#           def wrapper(state, **kwargs):
#               last_exception = None
#               for attempt in range(max_retries + 1):
#                   try:
#                       return node_fn(state, **kwargs)
#                   except RETRIABLE_EXCEPTIONS as exc:
#                       last_exception = exc
#                       if attempt < max_retries:
#                           delay = min(base_delay * (backoff_factor ** attempt), max_delay)
#                           time.sleep(delay)
#                   except Exception as exc:
#                       return {"error": f"[{node_fn.__name__}] {type(exc).__name__}: {exc}",
#                               "current_step": node_fn.__name__}
#               return {"error": f"[{node_fn.__name__}] 모든 재시도 실패: {last_exception}",
#                       "current_step": node_fn.__name__}
#           return wrapper
#       return decorator

# Retriable 예외 목록 (실제로는 anthropic, httpx 예외 사용)
RETRIABLE_EXCEPTIONS = (ConnectionError, TimeoutError)


def retry_on_llm_error(
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    backoff_factor: float = 2.0,
):
    """LLM 호출 노드용 재시도 데코레이터."""
    def decorator(node_fn):
        @wraps(node_fn)
        def wrapper(state, **kwargs):
            # TODO: max_retries + 1번 시도하는 루프
            # TODO: RETRIABLE_EXCEPTIONS → 재시도
            # TODO: 그 외 예외 → {"error": ..., "current_step": ...} 반환
            # TODO: 모든 재시도 실패 → {"error": ..., "current_step": ...} 반환
            pass  # 이 줄을 지우고 구현하세요
        return wrapper
    return decorator


print("retry_on_llm_error 데코레이터 정의 완료")

In [ ]:
# 검증 셀: Non-retriable 에러 → state["error"] 반환
class TestState(TypedDict):
    query: str
    result: str | None
    error: str | None
    current_step: str

@retry_on_llm_error(max_retries=2, base_delay=0.01)
def node_with_non_retriable_error(state: TestState) -> dict:
    raise ValueError("잘못된 입력")  # Non-retriable

result = node_with_non_retriable_error({"query": "test", "result": None, "error": None, "current_step": "start"})

assert result is not None, "Non-retriable 에러도 None이 아닌 dict를 반환해야 합니다"
assert "error" in result, "반환 dict에 'error' 키가 있어야 합니다"
assert result["error"] is not None, "error가 None이 아니어야 합니다"
print(f"Non-retriable 에러 처리 통과: {result['error'][:60]}...")

In [ ]:
# 검증 셀: Retriable 에러 → 재시도 후 성공
retry_call_count = 0

@retry_on_llm_error(max_retries=3, base_delay=0.01)
def node_with_retriable_error(state: TestState) -> dict:
    global retry_call_count
    retry_call_count += 1
    if retry_call_count < 3:
        raise ConnectionError("일시적 연결 실패")  # Retriable
    return {"result": "성공", "current_step": "done"}

retry_call_count = 0
result = node_with_retriable_error({"query": "test", "result": None, "error": None, "current_step": "start"})

assert result.get("result") == "성공", f"3번째 시도에서 성공해야 합니다. 현재: {result}"
assert retry_call_count == 3, f"3번 시도해야 합니다. 현재: {retry_call_count}번"
print(f"Retriable 에러 재시도 통과: {retry_call_count}번 시도 후 성공")
print("실습 1 완료!")

## 실습 2: 3노드 에이전트 구성

`fetch_data → analyze → create_ticket` 3노드 에이전트에  
Retry 데코레이터와 Circuit Breaker를 통합합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): TaskState TypedDict를 정의하고, 3개 노드 함수를 작성합니다.
#               - fetch_data_node: Circuit Breaker로 외부 API 호출 모방
#               - analyze_node: @retry_on_llm_error + FakeLLM 호출
#               - create_ticket_node: Circuit Breaker로 티켓 생성 모방
#
# 힌트 2 (키워드): TypedDict, @retry_on_llm_error(max_retries=2, base_delay=0.01),
#               api_circuit.call(), FakeLLM(responses={...}).invoke()
#
# 힌트 3 (거의 정답):
#   class TaskState(TypedDict):
#       query: str
#       fetched_data: str | None
#       analysis: str | None
#       ticket_key: str | None
#       error: str | None
#       current_step: str
#
#   api_circuit = CircuitBreaker("api", failure_threshold=5, expected_exceptions=(ConnectionError,))
#   ticket_circuit = CircuitBreaker("ticket", failure_threshold=5, expected_exceptions=(ConnectionError,))
#   llm = FakeLLM(responses={"분석": "버그 발견: 인증 로직 누락"})

# 1. 상태 정의
class TaskState(TypedDict):
    query: str
    fetched_data: str | None
    analysis: str | None
    ticket_key: str | None
    error: str | None
    current_step: str


# 2. Circuit Breaker 및 FakeLLM 설정
api_circuit = CircuitBreaker("api", failure_threshold=5, expected_exceptions=(ConnectionError,))
ticket_circuit = CircuitBreaker("ticket", failure_threshold=5, expected_exceptions=(ConnectionError,))
llm = FakeLLM(responses={"분석": "버그 발견: 인증 로직 누락", "코드": "코드 분석 완료"})


# 3. 노드 함수 구현
def fetch_data_node(state: TaskState) -> dict:
    """외부 API에서 데이터를 가져오는 노드 (Circuit Breaker 적용)."""
    try:
        # TODO: api_circuit.call()로 외부 API 호출 모방
        # (성공 케이스: f"데이터 수집 완료: {state['query']}" 반환)
        data = None  # TODO
        return {"fetched_data": data, "current_step": "fetch_data"}
    except CircuitOpenError as exc:
        return {"error": f"API 서킷 열림: {exc}", "current_step": "fetch_data"}
    except Exception as exc:
        return {"error": f"데이터 가져오기 실패: {exc}", "current_step": "fetch_data"}


@retry_on_llm_error(max_retries=2, base_delay=0.01)
def analyze_node(state: TaskState) -> dict:
    """LLM으로 분석하는 노드 (Retry 데코레이터 적용)."""
    if not state.get("fetched_data"):
        raise ValueError("분석할 데이터 없음")  # Non-retriable
    # TODO: llm.invoke()로 분석 수행
    result = None  # TODO: llm.invoke(f"분석: {state['fetched_data']}").content
    return {"analysis": result, "current_step": "analyze"}


def create_ticket_node(state: TaskState) -> dict:
    """티켓을 생성하는 노드 (Circuit Breaker 적용)."""
    try:
        # TODO: ticket_circuit.call()로 티켓 생성 모방
        # (성공 케이스: "TICKET-001" 반환)
        key = None  # TODO
        return {"ticket_key": key, "current_step": "create_ticket"}
    except CircuitOpenError as exc:
        return {"error": f"티켓 서킷 열림: {exc}", "current_step": "create_ticket"}
    except Exception as exc:
        return {"error": f"티켓 생성 실패: {exc}", "current_step": "create_ticket"}


def handle_error_node(state: TaskState) -> dict:
    """에러를 처리하는 노드."""
    print(f"[에러 처리] {state.get('error')}")
    return {"current_step": "handle_error"}


print("노드 함수 정의 완료")

In [ ]:
# TODO: 에러 라우팅 함수 구현
#
# 힌트 1 (방향): state에 "error" 키가 있고 None이 아니면 "handle_error"로,
#               그렇지 않으면 다음 정상 노드로 라우팅합니다.
#
# 힌트 2 (키워드): state.get("error"), return "handle_error" or "next_node"
#
# 힌트 3 (거의 정답):
#   def route_after_fetch(state: TaskState) -> str:
#       return "handle_error" if state.get("error") else "analyze"
#   def route_after_analyze(state: TaskState) -> str:
#       return "handle_error" if state.get("error") else "create_ticket"

def route_after_fetch(state: TaskState) -> str:
    """fetch_data 후 에러 여부로 라우팅."""
    pass  # TODO: error 있으면 "handle_error", 없으면 "analyze"


def route_after_analyze(state: TaskState) -> str:
    """analyze 후 에러 여부로 라우팅."""
    pass  # TODO: error 있으면 "handle_error", 없으면 "create_ticket"


print("라우팅 함수 정의 완료")

In [ ]:
# 그래프 구성
def build_resilient_graph():
    graph = StateGraph(TaskState)

    graph.add_node("fetch_data", fetch_data_node)
    graph.add_node("analyze", analyze_node)
    graph.add_node("create_ticket", create_ticket_node)
    graph.add_node("handle_error", handle_error_node)

    graph.set_entry_point("fetch_data")

    graph.add_conditional_edges(
        "fetch_data",
        route_after_fetch,
        {"analyze": "analyze", "handle_error": "handle_error"},
    )
    graph.add_conditional_edges(
        "analyze",
        route_after_analyze,
        {"create_ticket": "create_ticket", "handle_error": "handle_error"},
    )
    graph.add_edge("create_ticket", END)
    graph.add_edge("handle_error", END)

    return graph.compile()


app = build_resilient_graph()
print("그래프 구성 완료")

In [ ]:
# 검증 셀: 정상 케이스 실행
initial_state = {
    "query": "로그인 API 코드를 분석해주세요",
    "fetched_data": None,
    "analysis": None,
    "ticket_key": None,
    "error": None,
    "current_step": "start",
}

result = app.invoke(initial_state)

print(f"최종 상태: {result['current_step']}")
print(f"에러: {result.get('error')}")
print(f"티켓: {result.get('ticket_key')}")
print(f"분석: {result.get('analysis')}")

assert result.get("error") is None, f"정상 케이스에서 에러가 없어야 합니다. 현재: {result.get('error')}"
assert result.get("fetched_data") is not None, "fetched_data가 있어야 합니다"
print("정상 케이스 통과!")
print("실습 2 완료! 모든 실습 완료")

## 정리

이번 실습에서 학습한 내용:

1. **@retry_on_llm_error 데코레이터**: LangGraph 노드에 재시도 로직을 투명하게 추가
2. **에러 종류별 처리**:
   - Retriable (ConnectionError 등) → backoff 재시도
   - Non-retriable (ValueError 등) → 즉시 `state["error"]` 설정
3. **Circuit Breaker 통합**: 외부 API 노드에 `circuit.call()` 적용
4. **에러 라우팅**: `state["error"]` 필드로 에러 경로 분기

**다음 모듈**: Module 09 - 외부 시스템 연동 (asyncio, Rate Limiting, Health Check)